# MMA / UFC fighter pose — YOLO training (Colab + Google Drive)

1. Install **Ultralytics** and dependencies  
2. **Mount Google Drive**  
3. Point to your **YOLO pose dataset** on Drive (folder containing `data.yaml`, `train/`, `valid/` or `val/`, etc.)  
4. **Train** on Colab GPU (checkpoints under `/content` for speed)  
5. Validate, **MAE/RMSE**, plots  
6. Save **best.pt**, metrics JSON, and plots to a folder on **Drive**  

**Before running:** upload your dataset to Drive (same layout as Ultralytics: `data.yaml` next to `train/images`, `train/labels`, …).

**Runtime:** `Runtime` → `Change runtime type` → **GPU**.

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics pandas matplotlib numpy pyyaml opencv-python-headless tqdm

## 2. Configuration — edit Drive paths

- **`DRIVE_DATASET_DIR`**: folder that contains **`data.yaml`** and your `train` / `valid` (or `val`) splits.  
- **`DRIVE_OUTPUT_DIR`**: where to save each run (`best.pt`, `metrics_summary.json`, `plots/`, `train_run/`).

In [ ]:
from __future__ import annotations

import json
import shutil
from datetime import datetime
from pathlib import Path

# --- Google Drive paths (after mount, /content/drive/MyDrive/...) ---
DRIVE_DATASET_DIR = Path(
    "/content/drive/MyDrive/mma-fighter-pose-estimation-dataset"
)
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/UFC-PoseEstimation-runs")

# Training (lower batch or use yolo11n-pose.pt on small GPUs)
MODEL_NAME = "yolo11s-pose.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 8
RUN_NAME = "mma_pose_colab"

# Fast local disk for Ultralytics training cache
WORK_ROOT = Path("/content/mma_pose_work")
TRAIN_PROJECT = WORK_ROOT / "ultra_runs"

## 3. Mount Google Drive

Authorize when prompted so `/content/drive/MyDrive/...` paths work.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 4. Dataset & output paths

Checks that `data.yaml` exists and that each split’s `images` folder has a sibling `labels` folder (case-insensitive names, e.g. `Train`/`Images`).

In [ ]:
import yaml


def _ci_child(parent: Path, logical_name: str) -> Path | None:
    """Case-insensitive child dir (Train vs train, Labels vs labels)."""
    if not parent.is_dir():
        return None
    want = logical_name.lower()
    for c in parent.iterdir():
        if c.is_dir() and c.name.lower() == want:
            return c
    return None


def verify_yolo_dataset(data_yaml: Path) -> None:
    root = data_yaml.parent.resolve()
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    dr = root
    if cfg.get("path"):
        p = Path(str(cfg["path"]))
        dr = p.resolve() if p.is_absolute() else (root / p).resolve()
    else:
        dr = root
    for key in ("train", "val", "valid", "test"):
        if key not in cfg or cfg[key] in (None, ""):
            continue
        rel = Path(str(cfg[key]))
        full = rel.resolve() if rel.is_absolute() else (dr / rel).resolve()
        assert full.is_dir(), f"Missing split images folder ({key}): {full}"
        if full.name.lower() == "images":
            labels = _ci_child(full.parent, "labels")
            assert labels and labels.is_dir(), f"Missing labels folder next to {full}"
    print("Dataset layout OK:", data_yaml)


DATASET_DIR = DRIVE_DATASET_DIR.expanduser().resolve()
DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.is_file(), (
    f"Missing {DATA_YAML}\n"
    "Set DRIVE_DATASET_DIR to the folder that contains data.yaml (and train/, valid/, …)."
)

OUTPUT_DIR = DRIVE_OUTPUT_DIR.expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

verify_yolo_dataset(DATA_YAML)
print("DATASET_DIR:", DATASET_DIR)
print("DATA_YAML:", DATA_YAML)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 5. Train YOLO pose

Weights and logs go under `/content/mma_pose_work/ultra_runs` (fast disk). The next section copies weights, plots, and JSON to `OUTPUT_DIR`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
)

RUN_DIR = TRAIN_PROJECT / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.is_file(), f"Missing {BEST_PT}"
print("Training done. best.pt:", BEST_PT)

## 6. Official validation metrics (Ultralytics)

In [ ]:
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ, split="val", plots=True)

val_dict = val_metrics.results_dict if hasattr(val_metrics, "results_dict") else dict(val_metrics)
print("Validation summary (excerpt):")
for k in sorted(val_dict.keys()):
    if "map" in k.lower() or "precision" in k.lower() or "recall" in k.lower() or "loss" in k.lower():
        print(f"  {k}: {val_dict[k]}")

## 7. Helper: MAE / RMSE vs ground truth (val split)

Matches each GT box to the highest-IoU prediction, then compares **visible** keypoints (`v > 0`) in **normalized** image coordinates.

In [ ]:
import re

import numpy as np
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

K = 17
KPT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]


def _dataset_root(data_yaml: Path) -> Path:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    p = cfg.get("path")
    if p:
        root = Path(p)
        return root if root.is_absolute() else (data_yaml.parent / root).resolve()
    return data_yaml.parent.resolve()


def _resolve_split_dir(root: Path, key: str, data_yaml: Path) -> Path:
    with open(data_yaml, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    rel = cfg.get(key) or cfg.get("val") or cfg.get("valid")
    if not rel:
        raise ValueError(f"No val/valid path in {data_yaml}")
    rel = str(rel)
    p = Path(rel)
    return p if p.is_absolute() else (root / p).resolve()


def cxcywhn_to_xyxyn(cx, cy, w, h):
    x1, y1 = cx - w / 2.0, cy - h / 2.0
    x2, y2 = cx + w / 2.0, cy + h / 2.0
    return np.array([x1, y1, x2, y2], dtype=np.float64)


def iou_xyxyn(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return float(inter / union) if union > 0 else 0.0


def read_pose_labels(label_path: Path) -> list[dict]:
    """Each dict: bbox (cx,cy,w,h), xy (17,2), vis (17,) int."""
    if not label_path.is_file():
        return []
    out = []
    for line in label_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 5 + K * 3:
            continue
        cls = int(float(parts[0]))
        cx, cy, w, h = map(float, parts[1:5])
        rest = list(map(float, parts[5 : 5 + K * 3]))
        xy = np.array(rest[0::3], dtype=np.float64).reshape(-1, 1)
        yy = np.array(rest[1::3], dtype=np.float64).reshape(-1, 1)
        vis = np.array(rest[2::3], dtype=np.int64)
        xy = np.hstack([xy, yy])
        out.append({"cls": cls, "bbox": (cx, cy, w, h), "xy": xy, "vis": vis})
    return out


def evaluate_keypoint_errors(model: YOLO, data_yaml: Path, max_images: int | None = None):
    root = _dataset_root(data_yaml)
    val_img_root = _resolve_split_dir(root, "val", data_yaml)
    lbl_root = val_img_root.parent / "labels"
    assert val_img_root.is_dir(), val_img_root
    assert lbl_root.is_dir(), lbl_root

    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    images = sorted([p for p in val_img_root.rglob("*") if p.suffix.lower() in exts])
    if max_images:
        images = images[:max_images]

    per_k_sq = [[] for _ in range(K)]
    per_k_abs = [[] for _ in range(K)]
    all_sq = []
    all_abs = []
    n_matched_instances = 0

    for img_path in tqdm(images, desc="MAE/RMSE val"):
        stem = img_path.stem
        if stem.endswith(".rf"):
            m = re.match(r"^(.*)\.rf\.[a-f0-9]+$", stem)
            stem_alt = m.group(1) if m else stem
        else:
            stem_alt = stem
        lbl_path = lbl_root / f"{stem}.txt"
        if not lbl_path.is_file():
            lbl_path = lbl_root / f"{stem_alt}.txt"
        gts = read_pose_labels(lbl_path)
        if not gts:
            continue

        results = model.predict(source=str(img_path), imgsz=IMGSZ, verbose=False)
        if not results:
            continue
        r = results[0]
        if r.boxes is None or len(r.boxes) == 0 or r.keypoints is None or len(r.keypoints) == 0:
            continue

        pred_xyxyn = r.boxes.xyxyn.cpu().numpy()
        pred_xyn = r.keypoints.xyn.cpu().numpy()

        for gt in gts:
            gt_xyxy = cxcywhn_to_xyxyn(*gt["bbox"])
            best_j = -1
            best_iou = 0.0
            for j in range(pred_xyxyn.shape[0]):
                iou = iou_xyxyn(gt_xyxy, pred_xyxyn[j])
                if iou > best_iou:
                    best_iou = iou
                    best_j = j
            if best_j < 0 or best_iou < 0.1:
                continue
            n_matched_instances += 1
            px = pred_xyn[best_j]
            gx = gt["xy"]
            vis = gt["vis"]
            for ki in range(K):
                if vis[ki] <= 0:
                    continue
                d = np.linalg.norm(px[ki] - gx[ki])
                per_k_sq[ki].append(d * d)
                per_k_abs[ki].append(d)
                all_sq.append(d * d)
                all_abs.append(d)

    def rmse(xs):
        return float(np.sqrt(np.mean(xs))) if xs else float("nan")

    def mae(xs):
        return float(np.mean(xs)) if xs else float("nan")

    per_k = {
        KPT_NAMES[i]: {"mae": mae(per_k_abs[i]), "rmse": rmse(per_k_sq[i]), "n": len(per_k_abs[i])}
        for i in range(K)
    }
    summary = {
        "overall_mae_normalized": mae(all_abs),
        "overall_rmse_normalized": rmse(all_sq),
        "n_keypoint_comparisons": len(all_abs),
        "n_matched_instances": n_matched_instances,
        "note": "Errors in normalized [0,1] image coordinates; multiply by image side for rough pixels.",
        "per_keypoint": per_k,
    }
    return summary


kp_summary = evaluate_keypoint_errors(best_model, DATA_YAML, max_images=None)
print(json.dumps({k: v for k, v in kp_summary.items() if k != "per_keypoint"}, indent=2))

## 8. Plots (10+ figures)

Saves PNGs under `OUTPUT_DIR/plots/` using `results.csv`, MAE/RMSE bars, error histogram, and sample predictions.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

results_csv = RUN_DIR / "results.csv"
df = pd.read_csv(results_csv) if results_csv.is_file() else None

fig_n = 0


def save_fig(name: str):
    global fig_n
    fig_n += 1
    path = PLOTS_DIR / f"{fig_n:02d}_{name}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved", path)


if df is not None and len(df) > 0:
    epochs = np.arange(1, len(df) + 1)

    def col(*candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    # Fig 1–4: core losses (4 subplots = 4 visualizations)
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    ax = axes.ravel()
    pairs = [
        (col("train/box_loss", "train/box_loss"), "Train box loss"),
        (col("train/pose_loss", "train/pose_loss"), "Train pose loss"),
        (col("val/box_loss"), "Val box loss"),
        (col("val/pose_loss"), "Val pose loss"),
    ]
    for i, (cname, title) in enumerate(pairs):
        if cname and cname in df.columns:
            ax[i].plot(epochs, df[cname].values)
            ax[i].set_title(title)
            ax[i].set_xlabel("epoch")
            ax[i].grid(True, alpha=0.3)
        else:
            ax[i].set_visible(False)
    plt.suptitle("Training / validation losses", y=1.02)
    save_fig("losses_box_pose")

    # Fig 5: mAP-style metrics if present
    metric_cols = [c for c in df.columns if "mAP" in c or "map" in c.lower()]
    metric_cols = [c for c in metric_cols if "/" in c][:6]
    if metric_cols:
        fig, axes = plt.subplots(1, len(metric_cols), figsize=(4 * len(metric_cols), 3.5))
        if len(metric_cols) == 1:
            axes = [axes]
        for ax, c in zip(axes, metric_cols):
            ax.plot(epochs, df[c].values)
            ax.set_title(c.replace("metrics/", ""))
            ax.set_xlabel("epoch")
            ax.grid(True, alpha=0.3)
        plt.suptitle("mAP / metrics vs epoch", y=1.05)
        save_fig("map_metrics")

    # Fig 6: precision / recall
    pr_cols = [c for c in df.columns if "precision" in c.lower() or "recall" in c.lower()][:4]
    if pr_cols:
        fig, axes = plt.subplots(2, 2, figsize=(10, 7))
        axes = axes.ravel()
        for i, c in enumerate(pr_cols):
            axes[i].plot(epochs, df[c].values, label=c)
            axes[i].set_title(c)
            axes[i].set_xlabel("epoch")
            axes[i].grid(True, alpha=0.3)
        for j in range(len(pr_cols), 4):
            axes[j].set_visible(False)
        plt.suptitle("Precision / recall", y=1.02)
        save_fig("precision_recall")

    # Fig 7: learning rate
    lr_c = col("lr/pg0", "train/lr0", "lr0")
    if lr_c and lr_c in df.columns:
        plt.figure(figsize=(7, 4))
        plt.plot(epochs, df[lr_c].values)
        plt.title("Learning rate schedule")
        plt.xlabel("epoch")
        plt.grid(True, alpha=0.3)
        save_fig("learning_rate")

    # Fig 8: combined train total-ish proxy
    tcols = [c for c in [col("train/box_loss"), col("train/pose_loss")] if c and c in df.columns]
    if tcols:
        plt.figure(figsize=(7, 4))
        for c in tcols:
            plt.plot(epochs, df[c].values, label=c)
        plt.legend()
        plt.title("Train losses overlay")
        plt.xlabel("epoch")
        plt.grid(True, alpha=0.3)
        save_fig("train_loss_overlay")
else:
    print("No results.csv found at", results_csv)

In [ ]:
# Fig 9–10: MAE / RMSE per keypoint
pk = kp_summary["per_keypoint"]
names = list(pk.keys())
maes = [pk[n]["mae"] for n in names]
rmses = [pk[n]["rmse"] for n in names]

plt.figure(figsize=(12, 5))
plt.bar(range(K), maes, color="steelblue")
plt.xticks(range(K), names, rotation=60, ha="right")
plt.ylabel("MAE (normalized)")
plt.title("Per-keypoint MAE (visible GT only)")
plt.grid(True, axis="y", alpha=0.3)
save_fig("per_keypoint_mae")

plt.figure(figsize=(12, 5))
plt.bar(range(K), rmses, color="darkorange")
plt.xticks(range(K), names, rotation=60, ha="right")
plt.ylabel("RMSE (normalized)")
plt.title("Per-keypoint RMSE (visible GT only)")
plt.grid(True, axis="y", alpha=0.3)
save_fig("per_keypoint_rmse")

# Fig 11: histogram of all point errors
root = _dataset_root(DATA_YAML)
val_img_root = _resolve_split_dir(root, "val", DATA_YAML)
lbl_root = val_img_root.parent / "labels"
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
val_images = sorted([p for p in val_img_root.rglob("*") if p.suffix.lower() in exts])[:400]
hist_errors = []
for img_path in tqdm(val_images, desc="Histogram errors"):
    stem = img_path.stem
    lbl_path = lbl_root / f"{stem}.txt"
    if not lbl_path.is_file():
        continue
    gts = read_pose_labels(lbl_path)
    if not gts:
        continue
    results = best_model.predict(source=str(img_path), imgsz=IMGSZ, verbose=False)
    if not results or results[0].boxes is None or len(results[0].boxes) == 0:
        continue
    r = results[0]
    pred_xyxyn = r.boxes.xyxyn.cpu().numpy()
    pred_xyn = r.keypoints.xyn.cpu().numpy()
    for gt in gts:
        gt_xyxy = cxcywhn_to_xyxyn(*gt["bbox"])
        best_j = int(np.argmax([iou_xyxyn(gt_xyxy, pred_xyxyn[j]) for j in range(len(pred_xyxyn))]))
        if iou_xyxyn(gt_xyxy, pred_xyxyn[best_j]) < 0.1:
            continue
        px = pred_xyn[best_j]
        for ki in range(K):
            if gt["vis"][ki] <= 0:
                continue
            hist_errors.append(float(np.linalg.norm(px[ki] - gt["xy"][ki])))

plt.figure(figsize=(7, 4))
if hist_errors:
    plt.hist(hist_errors, bins=40, color="seagreen", edgecolor="white")
else:
    plt.text(0.5, 0.5, "No errors collected", ha="center", va="center")
plt.xlabel("L2 error (normalized)")
plt.ylabel("count")
plt.title("Distribution of per-keypoint errors (val, visible GT)")
plt.grid(True, alpha=0.3)
save_fig("error_histogram")

# Fig 12–15: sample prediction panels
import cv2

if len(val_images) >= 4:
    step = max(1, len(val_images) // 8)
    sample_paths = val_images[::step][:8]
else:
    sample_paths = list(val_images)
if len(sample_paths) < 4 and val_images:
    sample_paths = val_images[:4]

for si, sp in enumerate(sample_paths[:4]):
    res = best_model.predict(source=str(sp), imgsz=IMGSZ, verbose=False)[0]
    im = res.plot()
    im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if len(im.shape) == 3 and im.shape[2] == 3 else im
    plt.figure(figsize=(8, 8))
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title(f"Val prediction overlay: {sp.name}")
    save_fig(f"sample_pred_{si+1}")

print(f"Total figure files saved under: {PLOTS_DIR} (count ~{fig_n})")

## 9. Save model + metrics (`OUTPUT_DIR` on Drive)

In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_run = OUTPUT_DIR / f"run_{stamp}"
out_run.mkdir(parents=True, exist_ok=True)

results_csv = RUN_DIR / "results.csv"

shutil.copy2(BEST_PT, out_run / "best.pt")
if results_csv.is_file():
    shutil.copy2(results_csv, out_run / "results.csv")

# Optional: full training folder (weights, args, curves)
shutil.copytree(RUN_DIR, out_run / "train_run", dirs_exist_ok=True)

def _json_scalar(v):
    try:
        if hasattr(v, "item"):
            return float(v.item())
        return float(v)
    except (TypeError, ValueError):
        return str(v)


metrics_bundle = {
    "timestamp": stamp,
    "data_yaml": str(DATA_YAML),
    "model_base": MODEL_NAME,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "ultralytics_val": {k: _json_scalar(v) for k, v in val_dict.items()},
    "keypoint_mae_rmse": kp_summary,
}

with open(out_run / "metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(metrics_bundle, f, indent=2, default=str)

print("Saved to:", out_run)
print(" - best.pt")
print(" - metrics_summary.json")
print(" - train_run/ (full Ultralytics export)")
print(" - ../plots/ (PNG figures)")

## Notes

- **Ultralytics** pose metrics are mainly **OKS / mAP** style; **MAE/RMSE** here are extra, in **normalized** coordinates.
- If **IoU matching** fails often, lower the threshold in `evaluate_keypoint_errors` or inspect bbox alignment.
- **`model.val(..., plots=True)`** writes extra plots into the local run folder; those are copied under `train_run/` in the final save step.
- Training uses **`/content`** for speed; only **results** are written to **Drive** (`OUTPUT_DIR`).
- **`data.yaml`** should use paths that work on Linux (or rely on `path:` + relative splits). Folder names like `Train`/`Images` are OK; verification is case-insensitive.